```text
* 활용문서 : 종합부동산세법
* ocr 을 돌려서 md 파일로 생성: ./documents/output_docs/real_estate.md
```

```text
                         ┌────────────────────┐
                    ┌───▶│ 과세표준           │───┐
                    │    │                │   │
                    │    └────────────────────┘   │
                    │                             │
┌─────────┐         │    ┌────────────────────┐   │    ┌────────────────┐
│  START  │─────────┼───▶│ 공제금액           │───┼───▶│ 과세표준 계산  │
└─────────┘         │    └────────────────────┘   │    │                │
                    │                             │    └───────┬────────┘
                    │    ┌────────────────────┐   │            │
                    └───▶│ 공정시장가액비율   │───┘            ▼
                         └────────────────────┘        ┌────────────┐
                                                       │ 세율계산    │
                                                       └────────────┘

```

In [29]:
"""
[계산 시, 필요한 로직]
- `주택 수` → 사용자 질문에 있을 것
- `과세표준`
    - 종합부동산세법→ `“주택의 공시가격을 합산한 금액에서 다음 각 호의 금액을 공제한 금액에 부동산 시장의 동향과 재정 여건 등을 정하는 **공정시장가액비율**을 곱한 금액으로 한다.”`
    - `주택의 공시가격을 합산한 금액` → 사용자 질문에 있음
    - `공제한 금액`  → 종합부동산 세법에 있음
        - 세법 : `“대통령령으로 정하는 공정시장가액비율을 곱한 금액으로 한다. 다만, 그 금액이 영보다 작은 경우에는 ~”` → 이걸 벡터 스토어 대상으로 retrieve 해줘야 함
    - `공정시장가액비율` → 대통령령이므로 웹 서치(법안 없음)

[개발 노드 목록]
    과세표준 계산
    공제액
    공정시장가액비율 (대통령령)
    과세표준 계산
    세율 계산
"""

'\n[계산 시, 필요한 로직]\n- `주택 수` → 사용자 질문에 있을 것\n- `과세표준`\n    - 종합부동산세법→ `“주택의 공시가격을 합산한 금액에서 다음 각 호의 금액을 공제한 금액에 부동산 시장의 동향과 재정 여건 등을 정하는 **공정시장가액비율**을 곱한 금액으로 한다.”`\n    - `주택의 공시가격을 합산한 금액` → 사용자 질문에 있음\n    - `공제한 금액`  → 종합부동산 세법에 있음\n        - 세법 : `“대통령령으로 정하는 공정시장가액비율을 곱한 금액으로 한다. 다만, 그 금액이 영보다 작은 경우에는 ~”` → 이걸 벡터 스토어 대상으로 retrieve 해줘야 함\n    - `공정시장가액비율` → 대통령령이므로 웹 서치(법안 없음)\n\n[개발 노드 목록]\n    과세표준 계산\n    공제액\n    공정시장가액비율 (대통령령)\n    과세표준 계산\n    세율 계산\n'

In [30]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph

class AgentState(TypedDict):
    query: str  # 사용자 질문
    answer: str  # 세율 계산

    tax_base_equation: str # 과세표준 계산 수식
    tax_deduction: str # 공제액
    market_ratio: str # 공정시장가액비율
    tax_base: str  # 과세표준 계산

graph_builder = StateGraph(AgentState)

#### 문서 임베딩 및 DB 적재

In [32]:
# loader활용 시 테이블 구조가 사라진다
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# langchain 의 text splitter : 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 100,
    separators=['\n\n', '\n']
)

loader = UnstructuredMarkdownLoader(
    "../documents/output_docs/real_estate_tax.md", 
    mode="elements", 
    strategy="fast")

# data = loader.load()
document_list = loader.load_and_split(text_splitter=text_splitter) 

In [33]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

In [34]:
# 리트리버 생성
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name='real_estate_tax_collection',
    persist_directory='../real_estate_tax_collection'   # 로컬에 설정한 경로로 데이터가 남음 (DB의 역할을 한다)
)

retriever = vector_store.as_retriever(search_kwargs={'k':3})  # 3개 문서 반환

In [35]:
##########################################
## 테스트 코드
##########################################
from collections import Counter

data = vector_store.get(include=["documents"])
documents = data["documents"]
content_counts = Counter(documents)
duplicate_groups = {
    content: count
    for content, count in content_counts.items()
    if count > 1
}

print(f"전체 레코드 수: {len(documents)}")
print(f"고유 문서 수: {len(content_counts)}")
print(f"중복된 문서 그룹 수: {len(duplicate_groups)}")
print(
    "중복으로 추가된 레코드 수:",
    sum(count - 1 for count in duplicate_groups.values()),
)

전체 레코드 수: 318
고유 문서 수: 99
중복된 문서 그룹 수: 99
중복으로 추가된 레코드 수: 219


#### LLM 선언

In [36]:
from langsmith import Client

client = Client()

In [37]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='exaone3.5:7.8b', 
    num_predict=512,
)


# 경량화된 모델
small_llm = ChatOllama(
    model = 'bge-m3:latest'  
)

# 프롬프트
rag_prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,
)

#### 노드 생성

In [38]:
"""
* 기존 방식: `retrieve 노드 -> generate 노드` 를 생성 
    ** retrieve 된 결과만을 기반으로 generate 했기 때문에 가능했었음
* 현재 방식: `retrieve 노드를 생성하는게 적합하지 않음
    ** 수식, 공제액, 공정시장가액비율을 사전에 알아야 하고, 이 3가지를 기반으로 generate를 하기 때문
    ** 질문에 따라서 다이나믹하게 retrieve를 하는 방식을 채택
"""

'\n* 기존 방식: `retrieve 노드 -> generate 노드` 를 생성 \n    ** retrieve 된 결과만을 기반으로 generate 했기 때문에 가능했었음\n* 현재 방식: `retrieve 노드를 생성하는게 적합하지 않음\n    ** 수식, 공제액, 공정시장가액비율을 사전에 알아야 하고, 이 3가지를 기반으로 generate를 하기 때문\n    ** 질문에 따라서 다이나믹하게 retrieve를 하는 방식을 채택\n'

In [39]:
query = '5억짜리 집 1채, 10억짜리 집 1채, 20억짜리 집 1채를 가지고 있을 때 세금을 얼마나 내나요?'

In [40]:
# 일반적으로 LLM은 언어모델이기 때문에 번역, 요약, 분석에 유용하다. 
# 그래서 프롬프트를 분석하고 요약해서 번역해줘 -> 일괄적으로 시키면 원하는 결과를 얻어내기 어렵다

# get_tax_base_equation({}) 가 가지고 온 정보를 가지고 수식 추출을 위한 프롬프트를 하나 더 생성
# tax_base_retrieval_chain 과 tax_base_equation_chain 으로 분리
# 하나는 문서 반환용, return 한 내용을 수식 변환용

In [47]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 과세표준 계산 수식

# retriever를 동적으로 활용
tax_base_retrieval_chain = (
    {'context' : retriever, 'question': RunnablePassthrough()} 
    | rag_prompt | llm | StrOutputParser()
)


##############################################################
## 체인 세분화
##############################################################
from langchain_core.prompts import ChatPromptTemplate

tax_base_equation_prompt = ChatPromptTemplate.from_messages([
    ('system', '사용자의 질문에서 과세표준을 계산하는 방법을 수식으로 나타내주세요.부연설명없이 수식만 리턴해주세요'),
    ('human', '{tax_base_equation_information}')
])


# tax_base_retrieval_chain 를 구동해서 나온 답변을 input 으로 넣어준다
tax_base_equation_chain = (
    {'tax_base_equation_information' : RunnablePassthrough()} 
    | tax_base_equation_prompt
    | llm
    | StrOutputParser()
)

tax_base_chain = (
    {'tax_base_equation_information': tax_base_retrieval_chain}
    | tax_base_equation_chain
)
##############################################################

def get_tax_base_equation(state: AgentState) -> str:
    tax_base_equation_question = '주택에 대한 종합부동산세 계산시 과세표준을 계산하는 방법을 수식으로 표현해서 알려주세요'
    tax_base_equation = tax_base_chain.invoke(tax_base_equation_question)
    return {'tax_base_equation':tax_base_equation}  # state 를 return 


In [48]:
get_tax_base_equation({})  # 테스트 호출

"""
{'tax_base_equation': '주택의 종합부동산세 과세표준은 공시가격 합산액에서 공제 항목을 뺀 후, 부동산 시장 상황과 지방 조정을 고려해 대통령령으로 정한 공정시장가액비율(60%~100% 범위)을 곱한 금액입니다. 만약 계산 결과가 0 이하라면 0으로 처리됩니다. 이후 이 과세표준에 해당 세율을 적용하여 최종 세액을 산출합니다.'}
"""

"\n{'tax_base_equation': '주택의 종합부동산세 과세표준은 공시가격 합산액에서 공제 항목을 뺀 후, 부동산 시장 상황과 지방 조정을 고려해 대통령령으로 정한 공정시장가액비율(60%~100% 범위)을 곱한 금액입니다. 만약 계산 결과가 0 이하라면 0으로 처리됩니다. 이후 이 과세표준에 해당 세율을 적용하여 최종 세액을 산출합니다.'}\n"

#### 공제액

In [ ]:
tax_deduction_chain = (
    {'context' : retriever, 'question': RunnablePassthrough()} 
    | rag_prompt 
    | llm 
    | StrOutputParser()
)


def get_tax_deduction(state:AgentState):
    tax_deduction_question = '주택에 대한 종합부동산세 계산시 공제금액을 알려주세요'
    tax_deduction = tax_deduction_chain.invoke(tax_deduction_question)
    return {'tax_deduction':tax_deduction}  
    

In [50]:
get_tax_deduction({})

{'tax_deduction': '주택 종합부동산세 계산 시 1세대 1주택자에게는 산출된 세액의 80% 범위 내에서 공제가 적용될 수 있습니다. 정확한 공제금액은 법 개정일에 따라 변동될 수 있으니, 최신 법률을 참고하시기 바랍니다. 구체적인 금액은 상황에 따라 달라질 수 있습니다.'}

#### 공정시장가액비율

In [53]:
from langchain_tavily import TavilySearch
from datetime import date

# 프롬프트 작성
tax_market_ratio_prompt = ChatPromptTemplate.from_messages([
    ('system', f'아래 정보를 기반으로 공정시장 가액비율을 계산해주세요\n\nContext:\n{{context}}'),
    ('human', '{query}')
])

tavily_search_tool = TavilySearch(
    max_results=5,
    search_depth = "advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
)

def get_market_ratio(state:AgentState):
    query= f'오늘날짜:({date.today()})에 해당하는 주택 공시가격 공정시장가액비율은 몇%인가요?'
    context = tavily_search_tool.invoke(query)
    print(f'context === {context}')
    tax_market_ratio_chain = (
        tax_market_ratio_prompt
        | llm
        | StrOutputParser()
    )
    market_ratio = tax_market_ratio_chain.invoke({'context':context, 'query':query})
    
    return {'market_ratio' : market_ratio} # result는 List 형태

In [ ]:
get_market_ratio({})

"""
{'market_ratio': '2026년 10월 10일 기준으로 주택의 공시가격에 적용되는 공정시장가액비율은 **45%**입니다. \n\n이 비율은 정부의 부동산 정책에 따라 변동될 수 있으므로, 가장 정확한 정보를 얻으시려면 해당 시점의 공식 발표나 관련 정부 웹사이트를 확인하시는 것이 좋습니다. 하지만 일반적으로 제시된 정보에 따르면, 2026년에 대해서는 이 비율이 적용되고 있습니다. \n\n추가로 궁금한 사항이 있으시다면 언제든지 알려주세요!'}
"""

context === {'query': '오늘날짜:(2026-10-10)에 해당하는 주택 공시가격 공정시장가액비율은 몇%인가요?', 'follow_up_questions': None, 'answer': 'According to the sources, the public fair market value rate for housing property in 2026 is 60%. For single-family homes with a public assessment price of 3 billion won or less, the rate is 43%; for those exceeding 3 billion won but not exceeding 6 billion won, the rate is 44%; and for those exceeding 6 billion won, the rate is 45%. This rate is applied to determine the taxable base for property tax, which is then multiplied by the property tax rate to calculate the final tax amount.', 'images': ['https://blogthumb.pstatic.net/MjAyNjA2MDdfMTAx/MDAxNzgwNzg1ODA0OTAw.obxsRYTfVYSSJO-75aG8HSFABcY3w5eEwjWgM2gHaCMg.4U39g6GpA4smSLnrMzF8GGXtzYCEDKrM8EuRwtp_Cukg.PNG/0860ca2e-4c71-48a8-80b4-cf9fa53382b2.png?type=w2', 'https://www.joseilbo.com/gisa_img_origin/17768184131776818413_rozzhj_origin.png', 'https://blogthumb.pstatic.net/MjAyNjA3MThfOTgg/MDAxNzg0MzU4ODgxNTg1.26FfV5gp-Lr_5y9q90

{'market_ratio': '2026년 10월 10일 기준으로 주택의 공시가격에 적용되는 공정시장가액비율은 **45%**입니다. \n\n이 비율은 정부의 부동산 정책에 따라 변동될 수 있으므로, 가장 정확한 정보를 얻으시려면 해당 시점의 공식 발표나 관련 정부 웹사이트를 확인하시는 것이 좋습니다. 하지만 일반적으로 제시된 정보에 따르면, 2026년에 대해서는 이 비율이 적용되고 있습니다. \n\n추가로 궁금한 사항이 있으시다면 언제든지 알려주세요!'}

#### 과세표준 계산

In [ ]:
# 프롬프트 계산
from langchain_core.prompts import PromptTemplate

tax_base_caculation_prompt = PromptTemplate.from_template("""
주어진 내용을 기반으로 과세표준을 계산해주세요

과세표준 계산 공식: {tax_base_equation}
공제금액: {tax_deduction}
공정시장가액비율: {market_ratio}
사용자 주택 공시각겨 정보: {query}""")

def calculate_tax_base(state: AgentState):
    # tax_base_equation: str # 과세표준 계산 수식
    # tax_deduction: str # 공제액
    # market_ratio: str # 공정시장가액비율
    # tax_base: str # 과세표준 계산

    tax_base_equation = state['tax_base_equation']
    tax_deduction = state['tax_deduction']
    market_ratio = state['market_ratio']
    query = state['query']

    # chaining
    tax_base_caculation_chain = (
        tax_base_caculation_prompt 
        | llm 
        | StrOutputParser()
    )

    # invoke
    tax_base = tax_base_caculation_chain.invoke({
        'tax_base_equation' : tax_base_equation,
        'tax_deduction' : tax_deduction,
        'market_ratio' : market_ratio,
        'query' : query
    })

    print(f'tax_base === {tax_base}')  # 로그 (셀 output 에 가독성있게 찍힌다)
    return {'tax_base': tax_base}

In [ ]:
# 테스트 코드
initial_state = {
    'query': query,
    'tax_base_equation': '과세표준 = (주택 공시가격 합산 - 공제금액) × 공정시장가액비율',
    'tax_deduction':'주택 종합부동산세 계산 시 1세대 1주택자에게는 산출된 세액의 80% 범위 내에서 공제가 적용될 수 있습니다. 정확한 공제금액은 법 개정일에 따라 변동될 수 있으니, 최신 법률을 참고하시기 바랍니다. 구체적인 금액은 상황에 따라 달라질 수 있습니다.',
    'market_ratio': '2024년 주택 공시가격의 공정시장가액비율은 60%입니다.'
}

calculate_tax_base(initial_state)